## Check GPU Availability

In [ ]:
!nvidia_smi

In [ ]:
# Set CUDA Device Number
DEVICE_NUM = 0
ADDITIONAL_GPU = 3

from os import environ
environ["CUDA_VISIBLE_DEVICES"] = ",".join([f"{i+DEVICE_NUM}" for i in range(ADDITIONAL_GPU + 1)])
environ["CUDA_VISIBLE_DEVICES"]

## Imports

In [ ]:
import torch
import torch.nn as nn
import random

from transformers import AdamW, get_linear_schedule_with_warmup, Qwen3ForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM

from torch.utils.data import DataLoader
from datasets import load_dataset

from tqdm.auto import tqdm
import wandb

from utils.adapter import Adapter

In [ ]:
if torch.cuda.is_available():
    if ADDITIONAL_GPU:
        device = torch.device("cuda")
    else:
        device = torch.device(f"cuda")
else:
    device = torch.device("cpu")
    DEVICE_NUM = -1
    
print(f"INFO: Using device - {device}" + (f":{DEVICE_NUM}" if ADDITIONAL_GPU else ""))

## Wandb setting

In [ ]:
PROJECT_NAME = "restoration_training"
RUN_NAME = "Qwen3_8B_adapter_restoration"

# WandB Initialization
wandb.init(project=PROJECT_NAME, name=RUN_NAME)

## Define Dataset

In [ ]:
dataset_id = ""

In [ ]:
dataset = load_dataset(dataset_id)

In [ ]:
dataset['train'][0]

In [ ]:
dataset['train'][0].keys()

## Load Model

In [ ]:
torch_dtype = torch.bfloat16

In [ ]:
base_model_id = "Qwen/Qwen3-8B"
base_model = Qwen3ForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch_dtype,
    devicep_map="auto"
    )
tokenizer = AutoTokenizer.from_pretrained(base_model_id, use_fast=False)

In [ ]:
adapt_model_id = "pretrained/Qwen/Qwen3-8B_adapter"
adapt_model = Qwen3ForCausalLM.from_pretrained(
    adapt_model_id,
    torch_dtype=torch_dtype,
    device_map="auto"
    )

In [ ]:
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for training.")
    base_model = nn.DataParallel(base_model)
    adapt_model = nn.DataParallel(adapt_model)

In [ ]:
base_model.eval()
for param in base_model.parameters():
    param.requires_grad = False
    
adapt_model.train()

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples["Text"], truncation=True, padding="max_length", max_length=512)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.map(lambda x: {**x, 'labels': x['input_ids']})

In [ ]:
split_dataset = tokenized_datasets["train"].train_test_split(test_size=0.1, seed=42)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=8)
eval_dataloader = DataLoader(eval_dataset, shuffle=False, batch_size=8)

## Training

In [ ]:
# Hyperparameters
EPOCHS = 10
LEARNING_RATE = 5e-5
ALPHA = 0.5
T = 0.3
NUM_TRAINING_STEPS = EPOCHS * len(train_dataloader)

# optimizer and Scheduler
optimizer = AdamW(adapt_model.parameters(), lr=LEARNING_RATE)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=NUM_TRAINING_STEPS
)

# Loss Functions
distillation_loss_fn = nn.KLDivLoss(reduction="batchmean")
laungage_modeling_loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

# Initialize Adapter
adapter_trainer = Adapter(
    adapt_model=adapt_model,
    base_model=base_model,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    distillation_loss_fn=distillation_loss_fn,
    lm_loss_fn=laungage_modeling_loss_fn,
    temperature=T,
    alpha=ALPHA,
    device=device,
    use_wandb=True
)

# Run
adapter_trainer.run_training(
    train_dataloader,
    num_epochs=EPOCHS,
    eval_dataloader=eval_dataloader
)

wandb.finish()

## Testing

In [ ]:
output_dir = "./adapt_model_distilled"
model_to_save = adapt_model.module if isinstance(adapt_model, nn.DataParallel) else adapt_model
model_to_save.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Adapted model saved successfully to {output_dir}")

base_model.eval()
adapt_model.eval()

user_question = "Explain the concept of knowledge distillation."
input_text = f"Question: {user_question} Answer:"
encoded_input = tokenizer(input_text, return_tensors="pt").to(device)
input_ids = encoded_input.input_ids

print(f"Original Text: {input_text}\n")
print("=== Base Model vs Adapted Model Independent Generation Comparison ===\n")

base_output_ids = input_ids.clone()
adapt_output_ids = input_ids.clone()

max_new_tokens = 100
mismatched_tokens = 0
total_tokens = 0

base_gen_tokens = []
adapt_gen_tokens = []

for _ in range(max_new_tokens):
    total_tokens += 1
    
    with torch.no_grad():
        base_outputs = base_model(input_ids=base_output_ids)
        base_logits = base_outputs.logits[:, -1, :]
        base_predicted_id = torch.argmax(base_logits, dim=-1).unsqueeze(-1)

        adapt_outputs = adapt_model(input_ids=adapt_output_ids)
        adapt_logits = adapt_outputs.logits[:, -1, :]
        adapt_predicted_id = torch.argmax(adapt_logits, dim=-1).unsqueeze(-1)

    base_output_ids = torch.cat([base_output_ids, base_predicted_id], dim=-1)
    adapt_output_ids = torch.cat([adapt_output_ids, adapt_predicted_id], dim=-1)

    # comparing and recording generated tokens (for metrics calculation)
    if adapt_predicted_id.item() != base_predicted_id.item():
        mismatched_tokens += 1
    
    base_token_str = tokenizer.decode(base_predicted_id.squeeze())
    adapt_token_str = tokenizer.decode(adapt_predicted_id.squeeze())
    base_gen_tokens.append(base_token_str)
    adapt_gen_tokens.append(adapt_token_str)

    # 5. if either model generates EOS token, stop generation
    if base_predicted_id.item() == tokenizer.eos_token_id or adapt_predicted_id.item() == tokenizer.eos_token_id:
        print("INFO: EOS token generated. Halting generation.")
        break

print("\n--- Generation Results ---")
print("Base Model Output:", "".join(base_gen_tokens))
print("Adapted Model Output:", "".join(adapt_gen_tokens))
print("\n--- Evaluation Metrics ---")
token_accuracy = ((total_tokens - mismatched_tokens) / total_tokens) * 100 if total_tokens > 0 else 0
print(f"Token Match Rate (Accuracy): {token_accuracy:.2f}%")
print(f"Total Generated Tokens: {total_tokens}, Mismatched Tokens: {mismatched_tokens}")
